# Experiment 1: Study of NLP Applications and NLP Tools
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Compare NLTK, spaCy, Hugging Face Transformers and AllenNLP (and cloud NLP platforms) on resume text and choose the tools for the Resume Analyzer.

Open in Colab: [Exp1_NLP_Tools.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp1_NLP_Tools.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

In [ ]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = []
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:  # model not installed yet (fresh Colab runtime)
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
    import importlib, site
    importlib.invalidate_caches(); importlib.reload(site)
    import en_core_web_sm
    nlp = en_core_web_sm.load()

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

In [ ]:
# Five fictional (synthetic) resumes used for the screenshots - one per domain
SAMPLE_IDS = {"Engineering": "ENG_06", "HR": "HR_06", "Finance": "FIN_07",
              "Healthcare": "HLT_04", "Agriculture": "AGR_02"}
FOOTER = "Synthetic resume - fictional person"
def resume_text(rid):
    """Raw text of a fictional resume, without the 'synthetic resume' footer line."""
    t = df.set_index("resume_id").loc[rid, "extracted_text"]
    return "\n".join(l for l in t.split("\n") if not l.startswith(FOOTER)).strip()

eng = "Performed finite element analysis in ANSYS to validate bracket stiffness, reducing part weight by 14%."
hlt = ("Clinical Research Coordinator at Kindwell Children's Hospital, Chennai since February 2023, "
       "coordinating a Phase III trial across 27 sites.")
fin = "Reconciled accounts payable in Tally ERP and prepared GST returns."

## 1. NLTK – tokenization, POS tagging, stemming, lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
toks = nltk.word_tokenize(eng)
print("Tokens :", toks)
print("POS    :", nltk.pos_tag(toks))
words = [t for t in toks if t.isalpha()]
print("Stems  :", [PorterStemmer().stem(w) for w in words])
print("Lemmas :", [WordNetLemmatizer().lemmatize(w.lower(), pos="v") for w in words])

## 2. spaCy – one pipeline: tokens, lemmas, POS, dependencies, entities

In [ ]:
doc = nlp(hlt)
table = pd.DataFrame([(t.text, t.lemma_, t.pos_, t.dep_, t.head.text) for t in doc],
                     columns=["token", "lemma", "POS", "dependency", "head"])
display(table.head(12))
print("Entities:", [(e.text, e.label_) for e in doc.ents])

## 3. Hugging Face Transformers – RoBERTa-base tokenizer and contextual embeddings

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
tok = AutoTokenizer.from_pretrained("roberta-base")
roberta = AutoModel.from_pretrained("roberta-base").eval()
enc = tok(fin, return_tensors="pt")
print("Sub-word tokens:", tok.convert_ids_to_tokens(enc["input_ids"][0]))
with torch.no_grad():
    out = roberta(**enc)
print("Contextual embeddings shape:", tuple(out.last_hidden_state.shape), "(batch, tokens, hidden size)")
print(f"Parameters: {sum(p.numel() for p in roberta.parameters()):,}")

## 4. Speed on the five sample resumes

In [ ]:
import time
texts = [resume_text(r) for r in SAMPLE_IDS.values()]
nltk.word_tokenize(texts[0]); nlp(texts[0]); tok(texts[0])          # warm-up
def timed(fn):
    t0 = time.perf_counter(); n = fn(); return n, (time.perf_counter() - t0) * 1000
n1, t1 = timed(lambda: sum(len(nltk.word_tokenize(t)) for t in texts))
n2, t2 = timed(lambda: sum(len(d) for d in nlp.pipe(texts)))
n3, t3 = timed(lambda: sum(len(tok(t, truncation=False)["input_ids"]) for t in texts))
print(f"NLTK word_tokenize    : {n1:5d} tokens in {t1:7.1f} ms")
print(f"spaCy full pipeline   : {n2:5d} tokens in {t2:7.1f} ms (also tags, parses, NER)")
print(f"RoBERTa BPE tokenizer : {n3:5d} sub-words in {t3:7.1f} ms")

## 5. Libraries and cloud platforms compared
AllenNLP is archived and pins old PyTorch versions, and the cloud APIs are billed and would send resume data to a third party, so they were studied, not executed.

In [ ]:
libs = pd.DataFrame([
    ("NLTK", "Teaching & classic NLP: tokenizers, stemmers, WordNet, CFG/chunk parsers", "Exp 3-5 preprocessing, stemming, CFG parsing"),
    ("spaCy", "Fast production pipeline: tokenizer, tagger, parser, NER, rule matchers", "Sentence split, lemmas, NER, PhraseMatcher, dependency parse"),
    ("HF Transformers", "Pretrained transformer models (BERT, RoBERTa...) + fine-tuning", "RoBERTa-base fine-tuned to classify resumes"),
    ("AllenNLP", "Research library on PyTorch (SRL, coreference); no longer actively developed", "Not used - only studied"),
], columns=["library", "strength", "use in our project"])
cloud = pd.DataFrame([
    ("Google Cloud Natural Language", "entities, sentiment, syntax, content classification", "API key / service account, billed"),
    ("AWS Comprehend", "entities, key phrases, sentiment, PII detection, custom classifier", "IAM credentials, billed"),
    ("Azure AI Language (Text Analytics)", "NER, PII redaction, key phrases, sentiment, custom classification", "endpoint + key, billed"),
], columns=["cloud platform", "main features", "access"])
display(libs); display(cloud)